# Project: Regional PV production estimation from open data

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Structure and format inspired by the course given by Alexandre Mathieu at the Solar Academy in 2024, "PV failure detection based on operational time series". Physics and data: pvlib, PVGIS, ODRE, RTE.</sub>

This notebook invites the students to estimate the hourly PV production of one region from open data.

It is broken down into two parts:

1. **AC power modelling**, individual: Estimation of the operating variables of one PV system per student, compared with its actual data.
2. **Regional fleet**, in pairs: Estimation of the production of the whole fleet of the region, compared with the production published by RTE, with the gap attributed to a cause.

Read the project instructions (pdf) for the deliverables, the naming convention of the csv files, the deadline and the grading scale.

***Indicate your pair number k below, as text, between quotes. Groups 9* and 11* write "9b" and "11b". The members and the region come from the course table.***

In [ ]:
pair_k = "1" # Your pair number k, for example "1", "12", "9b" or "11b"

In [ ]:
# Import packages here
import numpy as np # For math operations
import pandas as pd # To create indexed time series
import matplotlib.pyplot as plt # To plot figures

# In Google Colab, pvlib is not installed by default: the line below installs it (takes 30 s)
!pip install -q pvlib
import pvlib
print("pvlib", pvlib.__version__)

# Those two lines avoid printing the warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# The region of the pair, from the online table
base = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/ENER958/main/project/"
pairs = pd.read_csv(f"{base}pairs.csv", dtype=str)
row = pairs.loc[pairs["pair"] == pair_k].iloc[0]
region = row["region"]
members = [int(row[c]) for c in ["student_a", "student_b", "student_c"] if isinstance(row[c], str) and row[c].strip() != ""]
print(f"Pair {pair_k}: region {region}, student numbers n = {members}")

# A. AC power modelling, one system per student

Each member runs this part on their own system. The input data follow the naming convention from pvlib:

- **ghi**, **dhi**, **dni**: irradiance [W/m²]
- **temp_air**: air temperature [°C]
- **wind_speed**: wind speed at 10 m [m/s]
- **solar_azimuth**, **solar_elevation**: sun position [°]

Actuals: **g_poa**, **t_mod**, **p_dc**, **p_ac**.

In [ ]:
def load_student(n):
    input_data = pd.read_csv(f"{base}data/student_{n}_inputs.csv", index_col=0)
    input_data.index = pd.to_datetime(input_data.index)
    pv_data = pd.read_csv(f"{base}data/student_{n}_actuals.csv", index_col=0)
    pv_data.index = pd.to_datetime(pv_data.index)
    system_data = pd.read_csv(f"{base}data/student_{n}_system.csv").iloc[0]
    return input_data, pv_data, system_data

# One entry per member: data[n] = (input_data, pv_data, system_data)
data = {n: load_student(n) for n in members}
for n in members:
    print(f"Student {n}:")
    print(data[n][2])

## A, member 1

Work on the system of the first member: `input_data, pv_data, system_data = data[members[0]]`.

In [ ]:
input_data, pv_data, system_data = data[members[0]]


### A.1. POA irradiance

In [ ]:
# Calculate the POA global irradiance and compare it with pv_data["g_poa"]


### A.2. Cell temperature model

In [ ]:
# Calculate the module temperature and compare it with pv_data["t_mod"]


### A.3. DC power model

In [ ]:
# Calculate the DC power and compare it with pv_data["p_dc"]


### A.4. AC power model

In [ ]:
# Calculate the AC power and compare it with pv_data["p_ac"]


### A.5. Performance metrics

- Performance ratio at the AC level
- Energy performance index

In [ ]:
# Calculate PR and EPI over the whole year and print them


### A.6. Export

In [ ]:
# ac_estimation = pd.DataFrame({"gpoa_estimated": ..., "t_mod_estimated": ..., "dc_power_estimated": ..., "ac_power_estimated": ...})
# ac_estimation.to_csv(f"student_{members[0]}_acestimation.csv")


## A, member 2

Same five tasks on the system of the second member. Reuse the functions written above. Skip this section if you work alone.

In [ ]:
if len(members) >= 2:
    input_data, pv_data, system_data = data[members[1]]
    # A.1 to A.5, then export student_{members[1]}_acestimation.csv


## A, member 3 (trios only)

Same five tasks on the system of the third member. Skip this section for a pair.

In [ ]:
if len(members) == 3:
    input_data, pv_data, system_data = data[members[2]]
    # A.1 to A.5, then export student_{members[2]}_acestimation.csv


# B. Regional fleet, per pair

In this section the pair estimates the hourly production of the PV fleet of the region:

0. Data check: the annual energy and the installed capacity rebuilt from the raw open data
1. Installed capacity by department, by size class and by commissioning date
2. A first model with one orientation, compared with RTE: what does the mean daily profile say?
3. A better representation of the fleet: classes with their shares, and the source of the shares
4. Comparison with RTE: one week, mean daily profile, monthly energy, metrics at three time scales
5. Error attribution, with one hypothesis tested
6. Bonus: how many weather nodes does the region need?

In [ ]:
reg = f"{base}regions/{region}/"
register = pd.read_csv(f"{reg}register.csv", dtype={"insee": str, "dept": str}, parse_dates=["commission_date"])
nodes = pd.read_csv(f"{reg}nodes.csv", dtype={"dept": str})
weather = {}
for dept in nodes["dept"]:
    w = pd.read_csv(f"{reg}weather_dept_{dept}.csv", index_col=0)
    w.index = pd.to_datetime(w.index).floor("h") # PVGIS labels the hours at HH:10, RTE at HH:00
    weather[dept] = w
reference = pd.read_csv(f"{reg}reference_2023.csv", index_col=0)
reference.index = pd.to_datetime(reference.index)
print(f"{region}: {len(register)} register rows, {round(register['kwp'].sum() / 1000)} MWp, {len(nodes)} departments, RTE {round(reference['ref_solar_mw'].sum() / 1000)} GWh in 2023")

### B.0. Data check

Rebuild the annual PV energy of your region in 2023 from the eCO2mix regional dataset, and the installed capacity from the national register, keeping the installations commissioned before 01/01/2024. Compare with the provided files and explain the gap. The links are in the project instructions.

In [ ]:
# Raw eCO2mix: time zone, half-hour step, sum to annual energy. Raw register: solar PV only, commissioned before 2024, capacity column.


### B.1. Installed capacity

In [ ]:
# Capacity by department, by size class, cumulative by commissioning date


### B.2. A first model, one orientation

In [ ]:
# Tilt 30, south, every department node, capacity-weighted sum. Compare with RTE and describe the mean daily profile.


### B.3. A better representation of the fleet

In [ ]:
# Classes with tilt, azimuth, share. State where the shares come from and what they changed.


### B.4. Comparison with RTE

In [ ]:
# One week, mean daily profile, monthly energy, metrics at hourly, daily and monthly scale


### B.5. Error attribution

In [ ]:
# Normalise both series by their annual energy, read the residual pattern, test one hypothesis and report the effect


### B.6. Bonus: weather nodes

In [ ]:
# One node against one node per department


### B.7. Export

Two csv files: `pair_k_region.csv` with one column `p_region_mw`, hourly, UTC, 2023, and `pair_k_classes.csv` with the columns `tilt`, `azimuth`, `share`.

In [ ]:
# p_region.rename("p_region_mw").to_csv(f"pair_{pair_k}_region.csv")
# classes.to_csv(f"pair_{pair_k}_classes.csv", index=False)
